# 01 — Environment Check

This notebook verifies the software environment for the **Adaptive_QEM_IBM** project before running the benchmark, noise, IBM hardware, and quantum error mitigation experiments.

**Checks:** Python version, Qiskit version, Qiskit Aer, IBM Runtime availability, a basic Bell-state circuit, local simulation, and basic circuit characterization.

## 1. Project Environment

Run this notebook first. If any required package is missing, fix the environment before continuing to the next notebook.

In [ ]:
import sys
import platform
import importlib.util

print('Python version :', sys.version)
print('Platform       :', platform.platform())
print('Python path    :', sys.executable)


In [ ]:
def package_version(package_name):
    try:
        module = __import__(package_name)
        return getattr(module, '__version__', 'version attribute unavailable')
    except Exception as exc:
        return f'NOT AVAILABLE: {exc}'

packages = ['qiskit', 'qiskit_aer', 'qiskit_ibm_runtime', 'numpy', 'scipy', 'pandas', 'matplotlib']

for package in packages:
    print(f'{package:20s}: {package_version(package)}')


## 2. Verify Required Imports

The project should use current Qiskit APIs rather than deprecated imports such as `from qiskit import Aer`.

In [ ]:
import qiskit
from qiskit import QuantumCircuit

print('Qiskit import: OK')
print('Qiskit version:', qiskit.__version__)

try:
    from qiskit_aer import AerSimulator
    print('Qiskit Aer import: OK')
except ImportError as exc:
    print('Qiskit Aer import: FAILED')
    print(exc)

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    print('Qiskit IBM Runtime import: OK')
except ImportError as exc:
    print('Qiskit IBM Runtime import: FAILED')
    print(exc)


## 3. Create a Basic Bell-State Circuit

The Bell state is used as the first sanity-check benchmark because it is small, deterministic at the measurement level, and contains a two-qubit interaction.

In [ ]:
bell = QuantumCircuit(2, 2, name='bell_phi_plus')
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])

print(bell)


In [ ]:
print('Qubits       :', bell.num_qubits)
print('Classical bits:', bell.num_clbits)
print('Depth        :', bell.depth())
print('Size         :', bell.size())
print('Gate counts  :', dict(bell.count_ops()))


## 4. Run the Bell Circuit on AerSimulator

This is a local ideal-simulation sanity check. The expected measurement outcomes for the Bell state are `00` and `11`.

In [ ]:
from qiskit_aer import AerSimulator

SHOTS = 1024
SEED_SIMULATOR = 42

simulator = AerSimulator()
job = simulator.run(
    bell,
    shots=SHOTS,
    seed_simulator=SEED_SIMULATOR
)
result = job.result()
counts = result.get_counts()

print('Bell-state counts:', counts)
print('Total shots      :', sum(counts.values()))


In [ ]:
correct = counts.get('00', 0) + counts.get('11', 0)
success_probability = correct / SHOTS

print(f'Bell success probability: {success_probability:.4f}')
print(f'Bell error probability  : {1 - success_probability:.4f}')


## 5. Optional IBM Quantum Runtime Check

This cell only checks whether an IBM Quantum Runtime service can be constructed from an already saved local account. It does **not** submit a job to hardware.

If no saved IBM Quantum account exists, the cell reports that hardware authentication is not configured and the notebook can still be considered successful for local development.

In [ ]:
try:
    from qiskit_ibm_runtime import QiskitRuntimeService

    try:
        service = QiskitRuntimeService()
        print('IBM Quantum Runtime account: AVAILABLE')
        print('Saved IBM Quantum service can be accessed.')
    except Exception as exc:
        service = None
        print('IBM Quantum Runtime account: NOT CONFIGURED/NOT ACCESSIBLE')
        print('Reason:', exc)
        print('This does not prevent local simulation.')

except ImportError as exc:
    service = None
    print('Qiskit IBM Runtime package is not installed.')
    print('Reason:', exc)


## 6. Optional IBM Backend Discovery

If IBM Runtime authentication is available, this cell lists accessible backends. No hardware execution is performed.

In [ ]:
if service is not None:
    try:
        backends = service.backends()
        print(f'Accessible backends: {len(backends)}')
        for backend in backends[:20]:
            print('-', backend.name)
    except Exception as exc:
        print('Backend discovery failed:', exc)
else:
    print('Skipped: IBM Quantum Runtime service is not available.')


## 7. Environment Status Summary

The following checks should be satisfied before proceeding to `02_ideal_benchmarks.ipynb`.

In [ ]:
import importlib.util

checks = {
    'Python available': sys.version_info >= (3, 10),
    'Qiskit available': importlib.util.find_spec('qiskit') is not None,
    'Qiskit Aer available': importlib.util.find_spec('qiskit_aer') is not None,
    'Bell circuit created': bell.num_qubits == 2 and bell.num_clbits == 2,
    'Aer simulation completed': sum(counts.values()) == SHOTS,
}

for name, status in checks.items():
    print(f"{'PASS' if status else 'FAIL':6s} | {name}")

required_pass = all(checks.values())
print('\nOverall local environment status:', 'READY' if required_pass else 'CHECK REQUIRED')
print('IBM hardware status:', 'AVAILABLE' if service is not None else 'NOT CONFIGURED/NOT VERIFIED')


## 8. Research Reproducibility Settings

These settings are recorded here and will be reused in later notebooks where applicable.

In [ ]:
EXPERIMENT_CONFIG = {
    'shots': 4096,
    'optimization_level': 3,
    'seed_simulator': 42,
    'seed_transpiler': 42,
}

for key, value in EXPERIMENT_CONFIG.items():
    print(f'{key:20s}: {value}')


### Next Step

After this notebook passes the local checks, proceed to **`02_ideal_benchmarks.ipynb`** to import the benchmark circuits from the `circuits/` package and generate the first reproducible benchmark dataset.